# Smart MCQ Solver — Inference Notebook
**Roll No**: 23f3002157 | **Notebook**: DL-23f3002157-notebook-t22026

Models: LSTM · GRU · BERT · DeBERTa · RAG Pipeline

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
kagglehub.dataset_download('risshabsrinivas/mcq-lstm-23f3002157 ')

In [ ]:
!pip install transformers sentence-transformers faiss-cpu torch tqdm -q

In [ ]:
import os, sys, json, pickle, string, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, CrossEncoder
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP   = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
IDX2LABEL   = {v: k for k, v in LABEL_MAP.items()}

TRAIN_PATH  = '/kaggle/input/smart-mcq-solver-challenge/train.csv'
TEST_PATH   = '/kaggle/input/smart-mcq-solver-challenge/test.csv'

LSTM_DIR    = '/kaggle/input/mcq-lstm-23f3002157'
GRU_DIR     = '/kaggle/input/mcq-gru-23f3002157'
BERT_DIR    = '/kaggle/input/mcq-bert-23f3002157'
DEBERTA_DIR = '/kaggle/input/mcq-deberta-23f3002157'
RAG_DIR     = '/kaggle/input/mcq-rag-23f3002157'

print(f'Device: {DEVICE}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f'Train: {train_df.shape} | Test: {test_df.shape}')
train_df.head(3)

## 3. EDA and Statistical Analysis

In [ ]:
# Answer distribution
freq = train_df['answer'].value_counts()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

freq.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].axhline(len(train_df)/5, color='red', linestyle='--', label='Uniform')
axes[0].set_title('Answer Distribution')
axes[0].legend()

train_df['prompt_len'] = train_df['prompt'].apply(lambda x: len(str(x).split()))
axes[1].hist(train_df['prompt_len'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Prompt Length Distribution')
axes[1].set_xlabel('Word Count')

for col in OPTION_COLS:
    axes[2].hist(train_df[col].apply(lambda x: len(str(x).split())),
                 bins=30, alpha=0.4, label=col)
axes[2].set_title('Option Length Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()
print(f'Answer counts:\n{freq}')

In [ ]:
# TF-IDF cosine similarity: prompt vs correct option
all_texts  = train_df['prompt'].tolist()
for col in OPTION_COLS:
    all_texts.extend(train_df[col].tolist())

tfidf_eda  = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_eda.fit(all_texts)

correct_sims, wrong_sims = [], []
for _, row in train_df.iterrows():
    pv = tfidf_eda.transform([str(row['prompt'])])
    for col in OPTION_COLS:
        ov  = tfidf_eda.transform([str(row[col])])
        sim = float(sk_cosine(pv, ov)[0][0])
        (correct_sims if col == row['answer'] else wrong_sims).append(sim)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(correct_sims, bins=40, alpha=0.6, color='green', label=f'Correct (mean={np.mean(correct_sims):.3f})')
ax.hist(wrong_sims,   bins=40, alpha=0.4, color='red',   label=f'Incorrect (mean={np.mean(wrong_sims):.3f})')
ax.set_title('TF-IDF Cosine Similarity: Correct vs Incorrect Options')
ax.legend()
plt.show()

# Hypothesis test
from scipy.stats import mannwhitneyu, chisquare
stat, p = mannwhitneyu(correct_sims, wrong_sims, alternative='greater')
print(f'Mann-Whitney U: stat={stat:.2f}, p={p:.6f}')
obs = freq.values
chi2, p2 = chisquare(obs, f_exp=np.full(5, len(train_df)/5))
print(f'Chi-square (uniform): chi2={chi2:.4f}, p={p2:.6f}')

## 4. Preprocessing

In [ ]:
def clean(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join([t for t in text.split() if t not in ENGLISH_STOP_WORDS])

proc_test = test_df.copy()
proc_test['prompt_clean'] = proc_test['prompt'].apply(clean)
for col in OPTION_COLS:
    proc_test[f'{col}_clean'] = proc_test[col].apply(clean)

print('Preprocessed test columns:', proc_test.columns.tolist())
print(f'Sample: {proc_test.iloc[0]["prompt_clean"][:80]}')

## 5. Basic Utils

In [ ]:
def top3_from_logits(logits_list):
    return [
        ' '.join([IDX2LABEL[i] for i in np.argsort(l)[::-1][:3]])
        for l in logits_list
    ]

def make_submission(ids, preds, filename):
    sub = pd.DataFrame({'id': ids, 'Prediction': preds})
    sub.to_csv(filename, index=False)
    print(f'Saved: {filename} ({len(sub)} rows)')
    return sub

def map_at_3(preds, labels):
    scores = []
    for p, l in zip(preds, labels):
        score = 0.0
        for rank, pred in enumerate(p[:3], 1):
            if pred == l:
                score = 1.0 / rank
                break
        scores.append(score)
    return round(float(np.mean(scores)), 4)



## 6. MODELS

### Model 1 (Scratch): LSTM

In [ ]:
sys.path.insert(0, LSTM_DIR)
from lstm_model import LSTMMCQModel

PAD_IDX = 0
UNK_IDX = 1

class Vocabulary:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
    def __len__(self):
        return len(self.word2idx)

with open(f'{LSTM_DIR}/vocab.json') as f:
    word2idx = json.load(f)

lstm_vocab = Vocabulary()
lstm_vocab.word2idx = word2idx

lstm_model = LSTMMCQModel(
    vocab_size=len(lstm_vocab), embed_dim=128,
    hidden_dim=256, num_layers=2, dropout=0.3, pad_idx=0
).to(DEVICE)
lstm_model.load_state_dict(
    torch.load(f'{LSTM_DIR}/best_model.pt', map_location=DEVICE)
)
lstm_model.eval()
print('LSTM loaded.')

In [ ]:
MAX_LEN_LSTM = 128

def encode_row_lstm(row, word2idx, max_len):
    prompt = str(row['prompt'])
    ids = []
    for col in OPTION_COLS:
        text   = (prompt + ' ' + str(row[col])).lower()
        text   = text.translate(str.maketrans('', '', string.punctuation))
        tokens = text.split()[:max_len]
        enc    = [word2idx.get(t, UNK_IDX) for t in tokens]
        enc   += [PAD_IDX] * (max_len - len(enc))
        ids.append(enc)
    return torch.tensor([ids], dtype=torch.long)

lstm_logits = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='LSTM'):
    inp = encode_row_lstm(row, word2idx, MAX_LEN_LSTM).to(DEVICE)
    with torch.no_grad():
        logits = lstm_model(inp)[0].cpu().numpy()
    lstm_logits.append(logits)

lstm_preds = top3_from_logits(lstm_logits)
print(f'Sample: {lstm_preds[0]}')

### Model 2 (Scratch) - GRU

In [ ]:
sys.path.insert(0, GRU_DIR)
from gru_model import GRUMCQModel

with open(f'{GRU_DIR}/vocab.json') as f:
    gru_word2idx = json.load(f)

gru_model = GRUMCQModel(
    vocab_size=len(gru_word2idx), embed_dim=192,
    hidden_dim=384, num_layers=3, num_heads=4, dropout=0.35, pad_idx=0
).to(DEVICE)
gru_model.load_state_dict(
    torch.load(f'{GRU_DIR}/best_model.pt', map_location=DEVICE)
)
gru_model.eval()
print('GRU loaded.')

In [ ]:
MAX_LEN_GRU = 160

def encode_row_gru(row, word2idx, max_len):
    prompt = str(row['prompt'])
    ids = []
    for col in OPTION_COLS:
        text   = (prompt + ' ' + str(row[col])).lower()
        text   = text.translate(str.maketrans('', '', string.punctuation))
        tokens = text.split()[:max_len]
        enc    = [word2idx.get(t, UNK_IDX) for t in tokens]
        enc   += [PAD_IDX] * (max_len - len(enc))
        ids.append(enc)
    return torch.tensor([ids], dtype=torch.long)

gru_logits = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='GRU'):
    inp = encode_row_gru(row, gru_word2idx, MAX_LEN_GRU).to(DEVICE)
    with torch.no_grad():
        logits = gru_model(inp)[0].cpu().numpy()
    gru_logits.append(logits)

gru_preds = top3_from_logits(gru_logits)
print(f'Sample: {gru_preds[0]}')

### Model 3 (Pre-Trained, FineTuned): BERT

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_DIR)

class BERTClassifier(nn.Module):
    def __init__(self, model_path, num_labels=5, dropout=0.1):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_path)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return self.classifier(cls)

bert_model = BERTClassifier(BERT_DIR).to(DEVICE)
bert_model.load_state_dict(
    torch.load(f'{BERT_DIR}/classifier.pt', map_location=DEVICE)
)
bert_model.eval()
print('BERT loaded.')

In [ ]:
MAX_LEN_BERT = 256
BATCH        = 16

class TransformerDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = [
            'Question: ' + str(r['prompt']) + ' ' +
            ' '.join([f'{c}: {str(r[c])}' for c in OPTION_COLS])
            for _, r in df.iterrows()
        ]
        self.tok = tokenizer
        self.max = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], max_length=self.max,
                       padding='max_length', truncation=True, return_tensors='pt')
        return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)

bert_ds     = TransformerDataset(test_df, bert_tokenizer, MAX_LEN_BERT)
bert_loader = DataLoader(bert_ds, batch_size=BATCH, shuffle=False)

bert_logits = []
for ids, mask in tqdm(bert_loader, desc='BERT'):
    with torch.no_grad():
        logits = bert_model(ids.to(DEVICE), mask.to(DEVICE)).cpu().numpy()
    bert_logits.extend(logits)

bert_preds = top3_from_logits(bert_logits)
print(f'Sample: {bert_preds[0]}')